In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import f1_score, classification_report
from sklearn.utils.class_weight import compute_class_weight
import numpy as np
import glob
import os
import matplotlib.pyplot as plt
from google.colab import drive

# ==========================================
# 1. AUTO-SETUP & CONFIG
# ==========================================
drive.mount('/content/drive', force_remount=True)
BASE_ROOT = "/content/drive/MyDrive/capture24"

print("🔍 Searching for processed research data...")
# Try multiple patterns to ensure we find the files
search_patterns = [
    os.path.join(BASE_ROOT, "processed_research", "*_research.npz"),
    os.path.join(BASE_ROOT, "**", "*_research.npz"), # Recursive fallback
]

found_files = []
for pattern in search_patterns:
    found_files = glob.glob(pattern, recursive=True)
    if found_files:
        print(f"   Found data using pattern: {pattern}")
        break

if not found_files:
    raise ValueError(f"❌ Critical Error: No '_research.npz' files found in {BASE_ROOT}. Please run the Data Processing step (Part 1) first!")

print(f"✅ Found {len(found_files)} participant files.")

# Hardware & Hyperparameters
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
FS = 100.0
WINDOW_LEN = 1000 # 10s
PATCH_SIZE = 20
PATCH_DIM = 60
NUM_PATCHES = 50
D_MODEL = 256
N_HEAD = 8
NUM_LAYERS = 6
CLASSES = ['sleep', 'sit-stand', 'walking', 'mixed', 'bicycling', 'vehicle', 'unknown']
NUM_CLASSES = len(CLASSES)

# Training Controls
BATCH_SIZE = 256
MAX_EPOCHS = 20          # Extended Training
PATIENCE = 5             # Early Stopping
LEARNING_RATE = 1e-4

# ==========================================
# 2. ADVANCED AUGMENTATIONS (Concern #1)
# ==========================================
class Augmentations:
    @staticmethod
    def apply(x):
        # 1. Jitter (Noise)
        if np.random.rand() < 0.5: x += torch.randn_like(x) * 0.05
        # 2. Scaling (Amplitude Shift)
        if np.random.rand() < 0.5: x *= (1.0 + torch.randn(1).to(x.device) * 0.1)
        # 3. Rotation (Sensor Shift)
        if np.random.rand() < 0.3:
            theta = (torch.rand(1) - 0.5) * np.pi / 2
            c, s = torch.cos(theta), torch.sin(theta)
            x_new = x.clone()
            x_new[..., 0] = x[..., 0]*c - x[..., 1]*s
            x_new[..., 1] = x[..., 0]*s + x[..., 1]*c
            x = x_new
        return x

class MultiTaskDataset(Dataset):
    def __init__(self, file_paths, mode='train'):
        self.mode = mode
        self.data_x = []
        self.data_y = []

        print(f"   Loading {len(file_paths)} files for {mode}...")
        for f in file_paths:
            try:
                d = np.load(f)
                x, y = d['x'], d['y']
                # Validate shape
                if x.shape[1] * x.shape[2] == NUM_PATCHES * PATCH_DIM:
                    x_patches = x.reshape(x.shape[0], NUM_PATCHES, PATCH_DIM)
                    self.data_x.append(x_patches)
                    self.data_y.append(y)
            except: pass

        if self.data_x:
            self.data_x = np.concatenate(self.data_x, axis=0)
            self.data_y = np.concatenate(self.data_y, axis=0)
        else:
            print(f"⚠️ Warning: {mode} dataset is empty!")
            self.data_x = np.zeros((0, NUM_PATCHES, PATCH_DIM))
            self.data_y = np.zeros((0,))

        self.data_x = torch.tensor(self.data_x, dtype=torch.float32)
        self.data_y = torch.tensor(self.data_y, dtype=torch.long)

    def __len__(self): return len(self.data_x)

    def __getitem__(self, idx):
        x = self.data_x[idx]
        y = self.data_y[idx]

        if self.mode == 'train':
            x_flat = x.view(-1, 3)
            x_aug = Augmentations.apply(x_flat)
            x = x_aug.view(NUM_PATCHES, PATCH_DIM)

        # SSL Mask (15%)
        ssl_mask = torch.rand(NUM_PATCHES) < 0.15

        # Forecast Mask (Last 10 patches = 1.0 second)
        fc_mask = torch.zeros(NUM_PATCHES, dtype=torch.bool)
        fc_mask[-10:] = True

        return x, ssl_mask, fc_mask, y

    def get_class_weights(self):
        labels = self.data_y.numpy()
        if len(labels) == 0: return torch.ones(NUM_CLASSES).to(DEVICE)

        # Balance weights: High weight for rare classes (Bicycling), Low for common (Unknown)
        classes = np.unique(labels)
        weights = compute_class_weight(class_weight='balanced', classes=classes, y=labels)

        weight_tensor = torch.ones(NUM_CLASSES)
        for cls, w in zip(classes, weights):
            if cls < NUM_CLASSES: weight_tensor[cls] = w

        return weight_tensor.to(DEVICE)

# ==========================================
# 3. RESEARCH MODEL (Concern #6 - MultiTask)
# ==========================================
class ResearchTransformer(nn.Module):
    def __init__(self):
        super().__init__()
        self.emb = nn.Linear(PATCH_DIM, D_MODEL)
        self.pos = nn.Parameter(torch.randn(1, NUM_PATCHES, D_MODEL) * 0.02)
        self.encoder = nn.TransformerEncoder(
            nn.TransformerEncoderLayer(D_MODEL, N_HEAD, 1024, 0.1, batch_first=True),
            NUM_LAYERS
        )
        self.head_ssl = nn.Linear(D_MODEL, PATCH_DIM)
        self.head_har = nn.Linear(D_MODEL, NUM_CLASSES)

    def forward(self, x):
        h = self.emb(x) + self.pos
        h = self.encoder(h)
        recon = self.head_ssl(h)
        gap = h.mean(dim=1)
        logits = self.head_har(gap)
        return recon, logits

# ==========================================
# 4. EXECUTION PIPELINE
# ==========================================
if __name__ == "__main__":
    # --- A. Participant Split (Concern #4) ---
    pids = sorted(list(set([os.path.basename(f).split('_')[0] for f in found_files])))
    split = int(0.8 * len(pids))
    train_pids = set(pids[:split])
    test_pids = set(pids[split:])

    train_files = [f for f in found_files if os.path.basename(f).split('_')[0] in train_pids]
    test_files = [f for f in found_files if os.path.basename(f).split('_')[0] in test_pids]

    print(f"🚀 Participants Split: {len(train_pids)} Train / {len(test_pids)} Test")

    # --- B. Load Data ---
    # Using subsets for quick demo execution. Remove slices [:X] for full run.
    train_ds = MultiTaskDataset(train_files[:50], mode='train')
    test_ds = MultiTaskDataset(test_files[:20], mode='test')

    if len(train_ds) > 0:
        train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=2)
        test_loader = DataLoader(test_ds, batch_size=BATCH_SIZE, shuffle=False)

        # --- C. Handle Imbalance (Concern #1) ---
        print("⚖️ Computing Class Weights...")
        class_weights = train_ds.get_class_weights()
        print(f"   Weights: {class_weights.cpu().numpy()}")

        # --- D. Training (Concern #2) ---
        model = ResearchTransformer().to(DEVICE)
        opt = optim.AdamW(model.parameters(), lr=LEARNING_RATE)
        scheduler = optim.lr_scheduler.ReduceLROnPlateau(opt, mode='min', factor=0.5, patience=2)

        crit_mse = nn.MSELoss()
        crit_ce = nn.CrossEntropyLoss(weight=class_weights) # Weighted Loss!

        print("\n--- 🚀 Starting Robust Multi-Task Training ---")
        best_loss = float('inf')
        patience_counter = 0

        for epoch in range(MAX_EPOCHS):
            model.train()
            l_sum = 0
            for x, m_ssl, _, y in train_loader:
                x, y = x.to(DEVICE), y.to(DEVICE)
                m_ssl = m_ssl.to(DEVICE)

                # Zero out masked patches
                x_in = x.clone()
                x_in[m_ssl] = 0

                recon, logits = model(x_in)

                # Multi-task Loss
                loss = crit_mse(recon[m_ssl], x[m_ssl]) + 0.5 * crit_ce(logits, y)

                opt.zero_grad()
                loss.backward()
                opt.step()
                l_sum += loss.item()

            avg_loss = l_sum/len(train_loader)
            scheduler.step(avg_loss)

            print(f"Epoch {epoch+1}/{MAX_EPOCHS} | Loss: {avg_loss:.4f} | LR: {opt.param_groups[0]['lr']:.2e}")

            # Early Stopping Check
            if avg_loss < best_loss:
                best_loss = avg_loss
                patience_counter = 0
                torch.save(model.state_dict(), "best_research_model.pth")
            else:
                patience_counter += 1
                if patience_counter >= PATIENCE:
                    print("🛑 Early Stopping: Model has converged.")
                    break

        # --- E. Detailed Evaluation (Concern #3 & #5) ---
        print("\n--- 📊 Final Research Evaluation ---")
        model.load_state_dict(torch.load("best_research_model.pth"))
        model.eval()

        y_true, y_pred = [], []
        err_05s, err_10s = [], []
        baseline_errs = [] # Persistence Baseline

        with torch.no_grad():
            for x, _, m_fc, y in test_loader:
                x, y = x.to(DEVICE), y.to(DEVICE)
                m_fc = m_fc.to(DEVICE)

                # Forecast
                x_in = x.clone()
                x_in[m_fc] = 0
                recon, logits = model(x_in)

                # 1. HAR
                y_true.extend(y.cpu().numpy())
                y_pred.extend(logits.argmax(1).cpu().numpy())

                # 2. Forecasting (Horizon Analysis)
                # m_fc is last 10 patches.
                # -5 is 0.5s horizon, -1 is 1.0s horizon (approx)
                diff = torch.abs(recon - x)

                # Ensure we have enough sequence length
                if diff.shape[1] >= 50:
                    err_05s.extend(diff[:, -5, :].mean(dim=1).cpu().numpy())
                    err_10s.extend(diff[:, -1, :].mean(dim=1).cpu().numpy())

                # 3. Baseline (Persistence)
                # Predicts that the last observed patch is repeated
                last_obs = x[:, -11, :].unsqueeze(1) # The patch right before mask
                baseline_diff = torch.abs(last_obs - x[:, -10:, :])
                baseline_errs.extend(baseline_diff.mean().cpu().numpy().flatten())

        print(f"\n1. Forecasting vs Baseline (MAE):")
        print(f"   Model @ 0.5s: {np.mean(err_05s):.4f}")
        print(f"   Model @ 1.0s: {np.mean(err_10s):.4f}")
        print(f"   Baseline Avg: {np.mean(baseline_errs):.4f}")
        print(f"   -> Improvement: {((np.mean(baseline_errs) - np.mean(err_10s)) / np.mean(baseline_errs))*100:.1f}%")

        print(f"\n2. HAR Classification Report (Known Classes Only):")
        # Filter Unknowns for clean report
        mask = np.array(y_true) != CLASS_MAP['unknown']
        print(classification_report(np.array(y_true)[mask], np.array(y_pred)[mask], target_names=CLASSES[:-1]))

        print("\n✅ Research-Grade Update Complete.")

Mounted at /content/drive
🔍 Searching for processed research data...
   Found data using pattern: /content/drive/MyDrive/capture24/processed_research/*_research.npz
✅ Found 151 participant files.
🚀 Participants Split: 120 Train / 31 Test
   Loading 50 files for train...
   Loading 20 files for test...
⚖️ Computing Class Weights...
   Weights: [ 0.59511006  0.5291464   3.3274317   1.8847187  30.251562    5.667375
  0.41855323]

--- 🚀 Starting Robust Multi-Task Training ---
Epoch 1/20 | Loss: 0.6569 | LR: 1.00e-04
Epoch 2/20 | Loss: 0.6042 | LR: 1.00e-04
Epoch 3/20 | Loss: 0.5875 | LR: 1.00e-04
Epoch 4/20 | Loss: 0.5756 | LR: 1.00e-04
Epoch 5/20 | Loss: 0.5666 | LR: 1.00e-04
Epoch 6/20 | Loss: 0.5585 | LR: 1.00e-04
Epoch 7/20 | Loss: 0.5526 | LR: 1.00e-04
Epoch 8/20 | Loss: 0.5457 | LR: 1.00e-04
Epoch 9/20 | Loss: 0.5414 | LR: 1.00e-04
Epoch 10/20 | Loss: 0.5365 | LR: 1.00e-04
Epoch 11/20 | Loss: 0.5314 | LR: 1.00e-04
Epoch 12/20 | Loss: 0.5274 | LR: 1.00e-04
Epoch 13/20 | Loss: 0.5217 |

NameError: name 'CLASS_MAP' is not defined